# Pet Breed Classifier — Edition 2 (final): Species-Split Models

**One EfficientNetV2-S per species, selected via class presets.**

Second and final edition of the training pipeline, matching our detector-based
inference: the detector decides cat vs dog, then routes the crop to the
corresponding 10-class fine-grained model. Set `CLASS_PRESET` and rerun to
produce each model — outputs are tagged per preset so runs never overwrite
each other (`best_model_cats.pth`, `best_model_dogs.pth`, ...).

Key findings from this edition:

- **Dogs reached ~99% test accuracy almost immediately; cats started at 72%.**
  Part of the gap is inherited from ImageNet-1k, which contains ~120 dog breed
  classes and only a handful of cats — the pretrained backbone arrives as a
  dog expert. Part is our data: *Tabby* and *Tiger_Cat* are coat patterns,
  not breeds, and permanently confuse each other and neighboring classes.
- **What fixed cats was data, not tricks:** after collecting additional images
  for underrepresented breeds and rebalancing, cat accuracy reached **95.4%**.
- The optional HuggingFace cat-breed backbone init (`CAT_HF_PRETRAIN`) was a
  **clean negative result**: the undocumented community checkpoint loaded
  correctly (780/782 tensors after unwrapping its `nn.Sequential` prefix) but
  gave no measurable gain. Kept here, disabled, for reproducibility.

In [ ]:
import os
import random
import time
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from PIL import Image
from tqdm import tqdm

## Class presets — the core of edition 2

In [ ]:
# Master list. Order defines ids for the "all" preset.
ALL_CLASSES = [
    "Abyssinian", "Bengal", "Birman", "Bombay", "British_Shorthair",
    "Maine_Coon", "Ragdoll", "Sphynx", "Tabby", "Tiger_Cat",
    "Beagle", "Pug", "Boxer", "Shiba_Inu", "Samoyed",
    "Golden_Retriever", "German_Shepherd", "Siberian_Husky", "Dalmatian", "Rottweiler",
]
CATS = ALL_CLASSES[:10]
DOGS = ALL_CLASSES[10:]

PRESETS = {"all": ALL_CLASSES, "cats": CATS, "dogs": DOGS}

CLASS_PRESET = "cats"    # <<< "all" / "cats" / "dogs" — rerun per model

CLASSES = PRESETS[CLASS_PRESET]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}  # ids are 0..N-1 within preset
NUM_CLASSES = len(CLASSES)

## Configuration

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/vaniakazakov/test-animals-rec/classes"
OUT_DIR = "/kaggle/working"
BATCH_SIZE = 64
TRAIN_SPLIT = 0.8
SEED = 42
IMG_SIZE = 300
NUM_WORKERS = 4

FROM_SCRATCH = False
EPOCHS = 80 if FROM_SCRATCH else 15
LR = 1e-3 if FROM_SCRATCH else 1e-4
WARMUP_EPOCHS = 5 if FROM_SCRATCH else 2

# Optional: initialize the cats backbone from a community HF checkpoint.
# Documented NEGATIVE result — loads cleanly, no measurable gain. Default off.
CAT_HF_PRETRAIN = False
CAT_HF_URL = ("https://huggingface.co/nabielherdiana/cat-breed-efficientnetv2"
              "/resolve/main/efficientnet_v2s_aug-model.pth")
CAT_HF_LOCAL = f"{OUT_DIR}/cat_effnetv2s_hf.pth"

use_cat_pretrain = (CAT_HF_PRETRAIN and CLASS_PRESET == "cats"
                    and not FROM_SCRATCH)

RUN_TAG = (f"{CLASS_PRESET}"
           f"{'_scratch' if FROM_SCRATCH else ''}"
           f"{'_hfcat' if use_cat_pretrain else ''}")
CKPT_PATH = f"{OUT_DIR}/best_model_{RUN_TAG}.pth"
CM_PATH = f"{OUT_DIR}/confusion_matrix_{RUN_TAG}.csv"

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPUS = torch.cuda.device_count() if DEVICE == "cuda" else 0
print(f"Device: {DEVICE} | GPUs: {N_GPUS}")
print(f"Preset: '{CLASS_PRESET}' -> {NUM_CLASSES} classes | run tag: {RUN_TAG}")

## Data

Only samples belonging to the active preset are used; ids are remapped to
`0..N-1` within the preset. Split is stratified per class.

In [ ]:
base = datasets.ImageFolder(DATASET_PATH)
missing = set(CLASSES) - set(base.classes)
assert not missing, f"Preset classes missing from dataset folders: {missing}"

active_folder_ids = {base.class_to_idx[c]: CLASS_TO_ID[c] for c in CLASSES}
all_samples = [(path, active_folder_ids[t]) for path, t in base.samples
               if t in active_folder_ids]
print(f"Samples in preset: {len(all_samples)} (dataset total: {len(base.samples)})")

rng = random.Random(SEED)
by_class = defaultdict(list)
for s in all_samples:
    by_class[s[1]].append(s)

train_samples, test_samples = [], []
for cls, items in by_class.items():
    rng.shuffle(items)
    n_train = int(TRAIN_SPLIT * len(items))
    train_samples.extend(items[:n_train])
    test_samples.extend(items[n_train:])
rng.shuffle(train_samples)

In [ ]:
NORM = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    NORM,
    transforms.RandomErasing(p=0.25),
])
test_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    NORM,
])


class PetDataset(Dataset):
    def __init__(self, samples, tf):
        self.samples = samples
        self.tf = tf

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert("RGB")
        return self.tf(img), label


train_ds = PetDataset(train_samples, train_tf)
test_ds = PetDataset(test_samples, test_tf)

# Weighted sampler: rare breeds are sampled as often as common ones
counts = Counter(lbl for _, lbl in train_samples)
wcls = {c: 1.0 / n for c, n in counts.items()}
sample_weights = [wcls[lbl] for _, lbl in train_samples]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         persistent_workers=True)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)} | Batches/epoch: {len(train_loader)}")
print("Per-class counts:", {CLASSES[c]: n for c, n in sorted(counts.items())})

## Optional HF cat-backbone loader (negative result, kept for reproducibility)

The checkpoint has no model card. The loader unwraps common container formats,
strips wrapper prefixes **including the `nn.Sequential` `0.` index this
checkpoint uses**, drops the foreign classifier head, guards against shape
mismatches, and reports exactly what loaded.

In [ ]:
def load_hf_cat_backbone(model, url=CAT_HF_URL, local_path=CAT_HF_LOCAL):
    if not os.path.exists(local_path):
        print("Downloading cat pretrain from HF...")
        torch.hub.download_url_to_file(url, local_path)

    ckpt = torch.load(local_path, map_location="cpu", weights_only=False)

    # 1) Unwrap container formats
    if isinstance(ckpt, nn.Module):
        state = ckpt.state_dict()
    elif isinstance(ckpt, dict):
        state = None
        for key in ("model_state", "state_dict", "model", "net", "weights"):
            if key in ckpt:
                inner = ckpt[key]
                state = inner.state_dict() if isinstance(inner, nn.Module) else inner
                break
        if state is None:
            state = ckpt
    else:
        raise TypeError(f"Unrecognized checkpoint type: {type(ckpt)}")

    # 2) Strip wrapper prefixes (incl. nn.Sequential '0.' — this checkpoint's case)
    for pref in ("module.", "model.", "backbone."):
        if state and all(k.startswith(pref) for k in state):
            state = {k[len(pref):]: v for k, v in state.items()}
    if any(k.startswith("0.") for k in state):
        state = {k[2:]: v for k, v in state.items() if k.startswith("0.")}

    # 3) Drop the foreign classifier head
    state = {k: v for k, v in state.items() if not k.startswith("classifier")}

    # 4) Shape guard
    own = model.state_dict()
    shape_dropped = [k for k, v in state.items()
                     if k in own and own[k].shape != v.shape]
    for k in shape_dropped:
        del state[k]

    # 5) Non-strict load onto ImageNet init + honest report
    result = model.load_state_dict(state, strict=False)
    n_total = len(own)
    n_loaded = n_total - len(result.missing_keys)
    print(f"[cat pretrain] loaded {n_loaded}/{n_total} tensors | "
          f"missing: {len(result.missing_keys)} | "
          f"unexpected: {len(result.unexpected_keys)} | "
          f"shape-dropped: {len(shape_dropped)}")
    if n_loaded < n_total * 0.8:
        print("[cat pretrain] WARNING: most keys did NOT load — model stays on ImageNet.")
    return n_loaded, n_total

## Model, loss, optimizer, schedule

In [ ]:
weights = None if FROM_SCRATCH else EfficientNet_V2_S_Weights.DEFAULT
model = efficientnet_v2_s(weights=weights)

if use_cat_pretrain:
    load_hf_cat_backbone(model)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)

if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel enabled on {N_GPUS} GPUs ({BATCH_SIZE // N_GPUS} imgs/GPU)")

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
# Warmup -> cosine: linear ramp from 10% LR, then cosine decay to ~0
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1,
                                          total_iters=WARMUP_EPOCHS),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                   T_max=EPOCHS - WARMUP_EPOCHS),
    ],
    milestones=[WARMUP_EPOCHS],
)
scaler = torch.amp.GradScaler() if DEVICE == "cuda" else None  # mixed precision

print(f"Run '{RUN_TAG}': "
      f"{'FROM SCRATCH' if FROM_SCRATCH else 'pretrained'}"
      f"{' + HF cat backbone' if use_cat_pretrain else ''} | "
      f"lr={LR} (warmup {WARMUP_EPOCHS} ep) | epochs={EPOCHS} | batch={BATCH_SIZE}")

## Metrics

In [ ]:
def macro_f1_and_cm(preds, labels):
    """Confusion matrix + macro F1 (accuracy alone lies under class imbalance)."""
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for p, t in zip(preds, labels):
        cm[t, p] += 1
    f1s = []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return float(np.mean(f1s)), f1s, cm


def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

## Training loop

In [ ]:
best_f1 = 0.0
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    cur_lr = optimizer.param_groups[0]["lr"]

    # --- train ---
    model.train()
    correct = total = 0
    running_loss = 0.0
    n_batches = 0
    pbar = tqdm(train_loader, desc=f"[{RUN_TAG}]  Epoch {epoch}/{EPOCHS}")
    for imgs, labels in pbar:
        imgs = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=scaler is not None):
            out = model(imgs)
            loss = criterion(out, labels)
        if scaler:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        running_loss += loss.item()
        n_batches += 1
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{running_loss / n_batches:.3f}",
                         acc=f"{100 * correct / total:.2f}%")
    train_loss = running_loss / n_batches
    train_acc = correct / total
    scheduler.step()

    # --- eval ---
    model.eval()
    all_preds, all_labels = [], []
    test_loss = 0.0
    n_test_batches = 0
    with torch.no_grad():
        for imgs, labels in tqdm(test_loader, desc="Testing", leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels_d = labels.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=scaler is not None):
                out = model(imgs)
                test_loss += criterion(out, labels_d).item()
            n_test_batches += 1
            all_preds.extend(out.argmax(1).cpu().tolist())
            all_labels.extend(labels.tolist())
    test_loss /= n_test_batches
    test_acc = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    f1, per_class_f1, cm = macro_f1_and_cm(all_preds, all_labels)
    dt = time.time() - t0

    # --- per-epoch report ---
    ranked = sorted(zip(CLASSES, per_class_f1), key=lambda x: x[1])
    worst = ", ".join(f"{n}={v:.2f}" for n, v in ranked[:3])
    best3 = ", ".join(f"{n}={v:.2f}" for n, v in ranked[-3:])
    print(f"\n===== EPOCH {epoch}/{EPOCHS} REPORT =====")
    print(f"  lr           : {cur_lr:.2e}")
    print(f"  train        : loss {train_loss:.4f} | acc {train_acc * 100:.2f}%")
    print(f"  test         : loss {test_loss:.4f} | acc {test_acc * 100:.2f}% | macro F1 {f1:.4f}")
    print(f"  worst classes: {worst}")
    print(f"  best classes : {best3}")
    print(f"  time         : {dt:.0f}s")

    if f1 > best_f1:
        best_f1 = f1
        torch.save({
            "model_state": unwrap(model).state_dict(),
            "classes": CLASSES,
            "epoch": epoch,
            "macro_f1": f1,
            "test_acc": test_acc,
            "from_scratch": FROM_SCRATCH,
            "img_size": IMG_SIZE,
            "preset": CLASS_PRESET,
            "cat_hf_pretrain": use_cat_pretrain,
        }, CKPT_PATH)
        np.savetxt(CM_PATH, cm, fmt="%d", delimiter=",",
                   header=",".join(CLASSES), comments="")
        print(f"  >> new best, checkpoint saved (macro F1 {f1:.4f})")
    print("=" * 44)

print(f"\nFinished. Best macro F1: {best_f1:.4f}")